What is Pydantic? Python is a "dynamically typed" language, meaning you can store a string in a variable that previously held an integer
. While this is easy for beginners, it causes "runtime errors" in production-level software because the data isn't predictable

Pydantic is a library used for data validation and data parsing
. It uses Python "type hints" to ensure that the data your program receives is valid and correctly typed before you process it
. Think of it as TypeScript for Python
.

Key Benefits:
* Type Safety: Ensures data follows a strict schema

* Automatic Coercion: It tries to convert data to the right type (e.g., string "10" to integer 10)

* Clear Errors: Provides detailed messages when validation fails

* Fast Performance: Built using Rust for high speed


In Pydantic, you define a Model by creating a class that inherits from BaseModel
. This defines the "Schema" or structure of your data


In [1]:
from pydantic import BaseModel, ValidationError
from typing import List, Dict, Optional
import json

In [2]:
class UserProfile(BaseModel):
    id: int           
    name: str         
    email: str        
    is_active: bool = True  

# Scenario 1: Valid Data
valid_data = {"id": 1, "name": "Vasanth", "email": "user1.@email.com"}
user = UserProfile(**valid_data)
print(f"User Created: {user.name}, Active: {user.is_active}")

User Created: Vasanth, Active: True


In [ ]:
invalid_data = {"id": "NotANumber", "name": "Vasanth"}
UserProfile(**invalid_data)

In [ ]:
# Scenario 2: Invalid Data (will raise an error)
try:
    invalid_data = {"id": "NotANumber", "name": "Vasanth"}
    UserProfile(**invalid_data)
except ValidationError as e:
    print("\nValidation Error caught!")
    print(e.json())

Type Coercion (The "Smart" Feature)


Pydantic is "smart"—if you send a string "30" to an integer field, it will automatically convert it for you
. However, if it cannot convert the data (like "abc" to an int), it throws an error

In [ ]:
class AgeModel(BaseModel):
    age: int

# Pydantic converts string "25" to integer 25 automatically
smart_user = AgeModel(age="25")
print(f"Coerced Age: {smart_user.age} (Type: {type(smart_user.age)})")

In [ ]:
smart_user = AgeModel(age="Twenty Five")
print(f"Coerced Age: {smart_user.age} (Type: {type(smart_user.age)})")

Complex Data Types (Lists & Dicts)

In real-world apps, data isn't just strings or ints; you often have lists (like tags) or dictionaries (like metadata)
Also, we use these data types from the typing module becasue using it gives us controll on the type of each element of that data type

In [ ]:
from typing import List, Dict

class Product(BaseModel):
    name: str
    tags: List[str]                # A list of strings
    metadata: Dict[str, str]       # A dictionary with string keys and values

product = Product(
    name="Laptop", 
    tags=["electronics", "tech"], 
    metadata={"brand": "Dell", "warranty": "2 years"}
)
print(f"Product Tags: {product.metadata}")

The Field Function for Constraints

What if an age cannot be negative, or a name must be at least 3 characters? The Field function allows you to add constraints and metadata


In [ ]:
from pydantic import Field

class Employee(BaseModel):
    # ID is required
    id: int
    # Name must be between 3 and 50 characters
    name: str = Field(..., min_length=3, max_length=50, description="Full name of employee")
    # Salary must be greater than 10,000
    salary: float = Field(..., gt=10000)
    # Department is optional with a default value
    dept: str = Field("General")

emp = Employee(id=101, name ="Jackob", salary=15000)
print(f"Employee: {emp.name}, Salary: {emp.salary}, Dept: {emp.dept}")

Field(...,  --> signifies that the field is required 

Custom Validation with @field_validator

Sometimes built-in constraints aren't enough. You can write your own logic using the @field_validator decorator


In [ ]:
from pydantic import BaseModel,field_validator

class Signup(BaseModel):
    email: str
    
    @field_validator('email')
    @classmethod
    def must_be_company_email(cls, v: str):
        if "wingerit.com" not in v:
            raise ValueError("Only wingerit.com emails are allowed")
        return v

try:
    user = Signup(email="fasial@wingerit.com")
except ValueError as e:
    print(f"Error: {e}")
else:
    print ("Valid User");

Model Validation (Multiple Fields)

Use @model_validator when validation depends on multiple fields (e.g., checking if password and confirm_password match)
.

In [4]:
from pydantic import model_validator

class Registration(BaseModel):
    password: str
    confirm_password: str

    @model_validator(mode="after") #validate a relationship between two fields after they're parsed
    def check_passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match!")
        return self

try:
    reg = Registration(password="123", confirm_password="123")
except ValueError as e:
    print(f"Error: {e}")
else:
    print("Access granted")

Access granted


In [5]:
from pydantic import BaseModel, model_validator

class User(BaseModel):
    first_name: str
    last_name: str

    @model_validator(mode="before") #transform or prepare multiple fields before validation happens
    def split_full_name(cls, data):
        if "full_name" in data:
            first, last = data["full_name"].split()
            data["first_name"] = first
            data["last_name"] = last
        return data


user = User(full_name="Shiv Kumar")
print(user)

first_name='Shiv' last_name='Kumar'


Nested Models

You can use one Pydantic model as a type inside another. This is great for organizing complex data like addresses
.

In [7]:
class Address(BaseModel):
    city: str
    zip_code: str

class Student(BaseModel):
    name: str
    home_address: Address # Nested Model

data = {
    "name": "Amit",
    "home_address": {"city": 3424, "zip_code": "302001"}
}

student = Student(**data)
print(f"Student: {student.name} lives in {student.home_address.city}")

ValidationError: 1 validation error for Student
home_address.city
  Input should be a valid string [type=string_type, input_value=3424, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

Serialization (Exporting Data)

Once you have validated data in an object, you often need to convert it back to a Dictionary or JSON for an API or database
.

In [8]:
# Convert model to Python Dictionary
user_dict = student.model_dump()
print(type(user_dict))
print(user_dict)

print("\n")

# Convert model to JSON String
user_json = student.model_dump_json()
print(type(user_json))
print(user_json)

print("\n")

# Export only specific fields
partial_dict = student.model_dump(include={"name"})
print(f"Include only Name: {partial_dict}")

<class 'dict'>
{'name': 'Amit', 'home_address': {'city': 'Jaipur', 'zip_code': '302001'}}


<class 'str'>
{"name":"Amit","home_address":{"city":"Jaipur","zip_code":"302001"}}


Include only Name: {'name': 'Amit'}


Handling Missing Data with Optional

In [10]:
from pydantic import BaseModel
from typing import Optional

class StudentProfile(BaseModel):
    name: str                   # Required
    email: str                  # Required
    phone_number: Optional[str] = None  # Non-mandatory, defaults to None

# Scenario: Student provides only name and email
data = {"name": "Arjun", "email": "arjun@college.edu"}
student = StudentProfile(**data)

print(f"Student: {student.name}")
print(f"Phone: {student.phone_number}") 

Student: Arjun
Phone: None


Metadata with Annotated

user registered
